# 08. 이벤트/공휴일 효과 분석

## 분석 배경 및 목적

공휴일, 명절, 연말연시 등 특수일(special days)은 택시 수요 패턴을 일상과 크게 다르게 만든다. 설/추석 연휴에는 도심 수요가 급감하는 반면 터미널/역 주변 수요가 급증하고, 연말연시에는 심야 수요가 폭증한다. 이러한 특수일 효과를 정량적으로 파악하면 수요 예측 모형의 정확도를 높이고, 운전자 수급 계획을 정교화할 수 있다.

본 분석은 공휴일/주말/평일 수요 차이, 명절 전후 패턴, 연말연시, 심야 수요를 다음 관점에서 분석한다.

1. **공휴일 효과**: 공휴일의 택시 수요는 평일/주말 대비 어떠한가?
2. **명절 전후 패턴**: 설/추석 D-3 ~ D+3 기간의 수요 곡선은 어떤 형태인가?
3. **연말연시 특수성**: 12/24~1/1 구간의 수요 패턴은 일반 공휴일과 어떻게 다른가?
4. **대체 공휴일 효과**: 대체 공휴일이 실제 수요에 미치는 영향은 일반 공휴일과 동일한가?

**방법론적 근거**: 공휴일 판정에는 한국천문연구원 특일 정보 API와 Python `holidays` 라이브러리를 활용하여 대체 공휴일, 임시 공휴일을 포함한 정확한 휴일 목록을 구성하였다. 특수일 효과 분석은 시계열 분해(seasonal decomposition)에서 holiday component를 분리하는 것과 동일한 논리이며, Facebook Prophet, SARIMAX 등 수요 예측 모형에서도 holiday regressor를 별도로 투입하는 것이 표준 관행이다.

> 공휴일은 하드코딩 대신 외부 데이터 `calendar_2018_2026.csv` / `holidays_2018_2026.csv`를 사용한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
import matplotlib.ticker as ticker
from datetime import timedelta

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. 외부 캘린더·공휴일 로드

In [ ]:
EXT_DIR = './external_data'
cal = pd.read_csv(f'{EXT_DIR}/calendar_2018_2026.csv', parse_dates=['date'])
hol = pd.read_csv(f'{EXT_DIR}/holidays_2018_2026.csv', parse_dates=['date'])
holiday_dates = set(hol['date'])

# 설날/추석 당일 (holiday_name 기준 추출)
def pick(names):
    mask = hol['holiday_name'].fillna('').str.contains('|'.join(names))
    return sorted(hol.loc[mask, 'date'])
# 명절 '당일'만 — '설날','추석'을 포함하되 전날/다음날/대체 제외
seollal = sorted(hol.loc[hol['holiday_name'].fillna('').str.contains('설날') &
                         ~hol['holiday_name'].fillna('').str.contains('전날|다음날|대체'), 'date'])
chuseok = sorted(hol.loc[hol['holiday_name'].fillna('').str.contains('추석') &
                         ~hol['holiday_name'].fillna('').str.contains('전날|다음날|대체'), 'date'])
print(f"공휴일 {len(holiday_dates)}일, 설날 {len(seollal)}개년, 추석 {len(chuseok)}개년")

## 2. 청크 집계: 일별 / 일별·시간대별

In [ ]:
D012_PATH = './DC_TBYXD012.csv'
usecols = ['RIDE_DTIME','PAY_AMT']
dtypes  = {'RIDE_DTIME': str,'PAY_AMT':'float64'}
daily, hourly = {}, {}
total = 0
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    rd = rd[rd.notna()]
    g = pd.DataFrame({'d': rd.dt.strftime('%Y%m%d'), 'h': rd.dt.hour})
    total += len(g)
    for k, v in g.groupby('d').size().items(): daily[k] = daily.get(k, 0) + v
    for k, v in g.groupby(['d','h']).size().items(): hourly[k] = hourly.get(k, 0) + v
    del chunk, rd, g; gc.collect()

daily_df = pd.Series(daily).rename_axis('d').reset_index(name='ride_count')
daily_df['date'] = pd.to_datetime(daily_df['d'], format='%Y%m%d')
daily_df = daily_df.drop(columns='d').sort_values('date').reset_index(drop=True)
daily_df['weekday'] = daily_df['date'].dt.dayofweek
hourly_df = pd.Series(hourly).rename_axis(['d','hour']).reset_index(name='count')
hourly_df['date'] = pd.to_datetime(hourly_df['d'], format='%Y%m%d')
hourly_df = hourly_df.drop(columns='d')
del daily, hourly; gc.collect()
print(f'전체 {total:,}건, 일수 {len(daily_df)}'); mem_usage('after load')

In [ ]:
# 일 유형 라벨 (공휴일 > 주말 > 평일)
def day_type(row):
    if row['date'] in holiday_dates: return '공휴일'
    if row['weekday'] >= 5: return '주말'
    return '평일'
daily_df['day_type'] = daily_df.apply(day_type, axis=1)
print(daily_df['day_type'].value_counts())

## 3. 공휴일 vs 평일 vs 주말

공휴일, 평일, 주말의 일평균 택시 수요를 비교한다. 공휴일은 법정 공휴일과 대체 공휴일을 포함하며, 주말과 겹치는 공휴일은 별도로 구분한다. 이 비교를 통해 공휴일의 수요 수준이 주말과 유사한지, 아니면 독립적인 효과를 갖는지를 판단할 수 있다. 이는 수요 예측 모형에서 공휴일을 주말과 동일하게 처리할지, 별도 변수로 투입할지를 결정하는 근거가 된다.

In [ ]:
stats = daily_df.groupby('day_type')['ride_count'].agg(['mean','median','std','count']).reindex(['평일','주말','공휴일'])
print(stats.round(0))
fig, ax = plt.subplots(figsize=(8, 5))
order = ['평일','주말','공휴일']; means = [stats.loc[t,'mean'] for t in order]
bars = ax.bar(order, means, color=['#4472C4','#ED7D31','#A5A5A5'], edgecolor='black', lw=0.5, width=0.5)
for b, v in zip(bars, means): ax.text(b.get_x()+b.get_width()/2, v, f'{v:,.0f}', ha='center', va='bottom')
ax.set_ylabel('평균 일일 승차건수'); ax.set_title('공휴일 vs 평일 vs 주말', fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 4. 명절 전후 (D-3 ~ D+3)

설/추석 당일을 기준으로 전후 3일간의 수요 변화를 분석한다. 명절 연휴는 일반 공휴일과 달리 수일에 걸쳐 수요 패턴이 점진적으로 변화하는 특성이 있다. D-1(전날)에는 귀성 수요로 터미널 주변 수요가 급증하고, 명절 당일에는 도심 수요가 최저점을 기록하며, D+1~D+2에는 귀경 수요가 발생한다. 이 비대칭적 패턴은 명절 기간 택시 배차 전략의 핵심 정보이다.

In [ ]:
def window(centers, w=3):
    rows = []
    for c in centers:
        for off in range(-w, w+1):
            r = daily_df[daily_df['date'] == c + timedelta(days=off)]
            if not r.empty: rows.append({'offset': off, 'ride_count': r['ride_count'].values[0]})
    if not rows: return pd.DataFrame()
    return pd.DataFrame(rows).groupby('offset')['ride_count'].mean().reset_index()

sp, cp = window(seollal), window(chuseok)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(axes, [sp, cp], ['설날','추석']):
    if data.empty: ax.set_title(f'{title} 데이터 없음'); continue
    ax.plot(data['offset'], data['ride_count'], 'o-', lw=2, color='#4472C4')
    ax.axvline(0, color='red', ls='--', alpha=0.5)
    ax.set_xticks(range(-3,4)); ax.set_xticklabels([f'D{i:+d}' if i else 'D-day' for i in range(-3,4)])
    ax.set_title(f'{title} 전후 수요'); ax.set_ylabel('평균 승차건수'); ax.grid(alpha=0.3, axis='y')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 5. 시간대별 프로파일 (공휴일/평일/주말)

In [ ]:
hd = hourly_df.merge(daily_df[['date','day_type']], on='date', how='left')
prof = hd.groupby(['day_type','hour'])['count'].mean().reset_index()
cmap = {'평일':'#4472C4','주말':'#ED7D31','공휴일':'#A5A5A5'}
fig, ax = plt.subplots(figsize=(12, 5))
for t in ['평일','주말','공휴일']:
    s = prof[prof['day_type']==t]
    ax.plot(s['hour'], s['count'], 'o-', ms=4, lw=2, label=t, color=cmap[t])
ax.set_xlabel('시간대'); ax.set_ylabel('평균 승차건수'); ax.set_title('시간대별 프로파일', fontweight='bold')
ax.set_xticks(range(24)); ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 6. 연말연시 (12/24 ~ 1/1)

크리스마스 이브부터 신년까지의 수요 패턴을 별도로 분석한다. 연말연시는 일반 공휴일과 달리 (1) 심야 수요가 극대화되고, (2) 수일간 연속으로 높은 수요가 유지되며, (3) 특정 지역(강남, 홍대, 이태원 등 유흥 지구)에 수요가 집중되는 특성이 있다. 이 기간은 택시 수급 불균형이 가장 심한 시기이므로 별도 관리가 필요하다.

In [ ]:
def is_ye(d): return (d.month==12 and d.day>=24) or (d.month==1 and d.day<=1)
ye = daily_df[daily_df['date'].apply(is_ye)].copy()
ye['order'] = ye['date'].apply(lambda d: d.day-24 if d.month==12 else 8)
ye['label'] = ye['date'].apply(lambda d: f'{d.month}/{d.day}')
avg = ye.groupby('order').agg(ride_count=('ride_count','mean'), label=('label','first')).reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg['order'], avg['ride_count'], 'o-', lw=2, color='#C00000')
ax.set_xticks(avg['order']); ax.set_xticklabels(avg['label'])
ax.set_title('연말연시 택시 수요', fontweight='bold'); ax.set_ylabel('평균 승차건수'); ax.grid(alpha=0.3, axis='y')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 7. 금/토/공휴일 전날 심야 수요

금요일, 토요일, 공휴일 전날의 심야(22시~04시) 수요를 일반 평일 심야와 비교한다. "내일 쉬는 날" 효과(tomorrow-off effect)로 심야 외출이 증가하면서 택시 수요가 급증하는 패턴을 정량화한다. 이 분석은 심야 택시 공급 계획과 심야할증 정책의 근거가 된다.

In [ ]:
eve = set()
for hd_ in holiday_dates:
    e = hd_ - timedelta(days=1)
    if e not in holiday_dates and e.weekday() < 5: eve.add(e)
late = hourly_df[hourly_df['hour'].isin([22,23,0,1,2,3])].merge(daily_df[['date','weekday']], on='date', how='left')
def lt(row):
    if row['date'] in eve: return '공휴일 전날'
    if row['weekday']==4: return '금요일'
    if row['weekday']==5: return '토요일'
    return None
late['late_type'] = late.apply(lt, axis=1)
late = late.dropna(subset=['late_type'])
daily_late = late.groupby(['date','late_type'])['count'].sum().reset_index()
late_avg = daily_late.groupby('late_type')['count'].mean()
fig, ax = plt.subplots(figsize=(8, 5))
labels = ['금요일','토요일','공휴일 전날']; vals = [late_avg.get(l,0) for l in labels]
bars = ax.bar(labels, vals, color=['#4472C4','#ED7D31','#A5A5A5'], edgecolor='black', lw=0.5, width=0.5)
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v, f'{v:,.0f}', ha='center', va='bottom')
ax.set_ylabel('평균 심야(22~03시) 승차건수'); ax.set_title('심야 수요 비교', fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 8. 요약

본 분석의 핵심 발견과 실무적 시사점을 정리한다.

**실무 활용**: 특수일 효과 분석 결과는 (1) 수요 예측 모형의 holiday regressor 설계, (2) 명절/연말 기간 택시 공급 확대 계획, (3) 심야 할증/배차 우선순위 정책에 직접 활용할 수 있다. 특히 명절 전후 수요 곡선은 고속버스/KTX 증편 계획과 연계하여 복합 교통 수급 관리에 활용 가능하다.

In [ ]:
wd = daily_df[daily_df['day_type']=='평일']['ride_count'].mean()
we = daily_df[daily_df['day_type']=='주말']['ride_count'].mean()
ho = daily_df[daily_df['day_type']=='공휴일']['ride_count'].mean()
print('=== 이벤트/공휴일 효과 요약 ===')
print(f"평일 {wd:,.0f} / 주말 {we:,.0f} ({we/wd:.0%}) / 공휴일 {ho:,.0f} ({ho/wd:.0%})")
for name, dates in [('설날', seollal), ('추석', chuseok)]:
    v = daily_df[daily_df['date'].isin(dates)]['ride_count']
    if not v.empty: print(f"{name} 당일 {v.mean():,.0f} (평일대비 {v.mean()/wd:.0%})")
for l in ['금요일','토요일','공휴일 전날']: print(f"심야 {l}: {late_avg.get(l,0):,.0f}")

---

## References

1. 한국천문연구원 (2024). 특일(공휴일) 정보 API. https://www.kasi.re.kr
2. Python `holidays` 라이브러리. https://github.com/vacanza/python-holidays (한국 공휴일, 대체 공휴일 지원)
3. Taylor, S. J., & Letham, B. (2018). Forecasting at Scale. *The American Statistician*, 72(1), 37-45. (Facebook Prophet - holiday component 방법론)
4. 서울시 (2024). 택시 수급 관리 계획. 서울특별시 교통정책과.